In [1]:
import sys
import numpy as np
import pandas as pd
import gamspy as gp

from src.parameters import *

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container(options=gp.Options(relative_optimality_gap=0))

/Users/Patron/Documents/cs524/.gamspy_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# LOAD DATA
teams_df = pd.read_csv("data/processed/hq.csv")
circuits_df = pd.read_csv("data/processed/circuits.csv")
distances_df = pd.read_csv("data/processed/distances.csv")
climate_df = pd.read_csv("data/processed/climate.csv")
# festivals_df = pd.read_csv("data/processed/festivals.csv")

In [3]:
# CONSTANTS
BREAK_START_DATE = 24 # 2026-08-09 (in climate_df)
BREAK_END_DATE = 26   # 2026-08-30 (in climate_df) 
BREAK_POINT = 14      # NO. OF RACES BEFROE BREAK

In [4]:
# PREPROCESSING

teams = ['ALL'] # teams_df["team_id"].tolist()[:-1]
circuits = circuits_df["circuit_id"].tolist()

weekends = climate_df['week_num'].astype(int).tolist()
summer_break = climate_df.loc[
    climate_df['week_num'].between(BREAK_START_DATE, BREAK_END_DATE), 
    'week_num'].astype(int).tolist()

race_emissions = distances_df[distances_df['from'].isin(circuits)][["from", "to", "emissions_kgCO2e"]]
hq_emissions = distances_df[distances_df['from'].isin(teams)][["from", "to", "emissions_kgCO2e"]]

In [5]:
n_teams = len(teams)
n_races = len(circuits)
n_weekends = len(weekends)

In [6]:
feasible_dates = np.zeros((n_weekends, n_races))
feasible_dates_df = pd.DataFrame(feasible_dates, columns=circuits, index=climate_df['week_num'].astype(int).tolist())

climate_df.index = climate_df['week_num'].astype(int).tolist()

for circuit in circuits:
    temp_col = f"{circuit}_avg_temp"
    precip_col = f"{circuit}_avg_precip"

    ok_temp = climate_df[temp_col].between(MIN_TEMP_F, MAX_TEMP_F)
    ok_precip = climate_df[precip_col] < MAX_PRECIP_IN

    feasible_dates_df.loc[ok_temp & ok_precip, circuit] = 1

    #Festival filter

In [7]:
# SET
Weekend = gp.Set(m, records=weekends)
SummerBreak = gp.Set(m, domain=[Weekend], records=summer_break)
Team = gp.Set(m, records=teams)
Circuit = gp.Set(m, records=circuits)
i = gp.Alias(m, alias_with=Circuit)
j = gp.Alias(m, alias_with=Circuit)
t = gp.Alias(m, alias_with=Weekend)



# PARAMETERS
r_emissions = gp.Parameter(m, domain=[Circuit, Circuit], records=race_emissions,
                         description="emissions between races in kgCO2e")
hq_emissions = gp.Parameter(m, domain=[Team, Circuit], records=hq_emissions,
                         description="emissions from HQ to race in kgCO2e")

feasible_dates = gp.Parameter(m, domain=[Weekend, Circuit], records=feasible_dates_df.stack().reset_index().values.tolist(),
                            description="feasibility of holding race on weekend (1 if feasible, 0 otherwise)")
feasible_dates[SummerBreak, Circuit] = 0 # No races during the break


first_set = gp.Parameter(m, records=14,
                         description='Number of races before the summer break')
break_start_date = gp.Parameter(m, records=np.array(BREAK_START_DATE),
                                description='Weekend where break begins')
break_end_date = gp.Parameter(m, records=np.array(BREAK_END_DATE),
                                description='Weekend where break ends')

In [8]:
# VARIABLES
x = gp.Variable(m,type='binary',domain=[Circuit,Circuit],
                description="1 if race in i is scheduled immediately before j, 0 otherwise")
y = gp.Variable(m, type='binary', domain=[Circuit,Weekend], 
                description='1 if race in Circuit is scheduled on Weekend, 0 otherwise')
z = gp.Variable(m, type="binary", domain=[i, j],
                description="1 if race i is on pre-break weekend and race j on post-break weekend")
u = gp.Variable(m,type='positive',domain=[Circuit], 
                description="Position of race in the calendar sequence")

x.fx[i,i] = 0

u.lo[Circuit] = 2                   # all races but first must be at least position 2
u.up[Circuit] = gp.Card(Circuit)    # all races must be at most position number of races

u.fx["BAH"] = 1                # First race = AUS
u.fx["ABD"] = gp.Card(Circuit) # Last race = ABD (position = number of races)



In [9]:
# EQUATIONS
# Obvious ones first
assign1 = gp.Equation(m, domain=[j],
                      description='Each circuit can be reached only from one other circuit')
assign1[j]= gp.Sum(i, x[i,j]) == 1

assign2 = gp.Equation(m, domain=[i],
                      description='Can only go to one other Circuit from each circuit')
assign2[i]= gp.Sum(j, x[i,j]) == 1

assign3 = gp.Equation(m, domain=[i],
                      description='Each race is only conducted once')
assign3[i]= gp.Sum(t, y[i,t]) == 1

assign4 = gp.Equation(m, domain=[t],
                      description='A weekend can conduct atmost 1 race')
assign4[t]= gp.Sum(i, y[i,t]) <= 1

mtz = gp.Equation(m,domain=[i,j],
                  description='This is the MTZ equation')
mtz[i,j].where[(i.ord > 1) & (j.ord > 1)]= (
  u[i] - u[j] + 1 <= (gp.Card(i) - 1) * (1 - x[i,j]))

# start_cons = gp.Equation(m)
# start_cons[...] = gp.Sum(i, x[i, "BAH"]) == 0

# end_cons = gp.Equation(m)
# end_cons[...] = gp.Sum(j, x["ABD", j]) == 0


In [10]:
# Time Constraints
time_cons1 = gp.Equation(m, domain=[i,t],
                         description='Race can be held only if the weekend is feasible')
time_cons1[i,t] = y[i,t] <= feasible_dates[t,i]

# time_cons2 = gp.Equation(m, domain=[i, j],
#     description="If j follows i, then weekend(j) must be after weekend(i)")
# time_cons2[i, j] = (
#     gp.Sum(t, t.ord * y[i, t]) -
#     gp.Sum(t, t.ord * y[j, t]) +
#     1 <= (gp.Card(i) - 1) * (1 - x[i, j])
# )

time_cons3 = gp.Equation(m,
                         description='Only K races until break')
time_cons3[...] = gp.Sum([i, t], y[i, t]*(t.ord < BREAK_START_DATE)) == first_set

# One race on the last weekend before break
pre_break_cons = gp.Equation(m)
pre_break_cons[...] = gp.Sum(Circuit, y[Circuit, BREAK_START_DATE - 1]) == 1

# One race on the first weekend after break
post_break_cons = gp.Equation(m)
post_break_cons[...] = gp.Sum(Circuit, y[Circuit, BREAK_END_DATE + 1]) == 1


In [11]:
# PARAMETERS identifying BAH and ABD
FIRST = "BAH"
LAST = "ABD"

# Identify last weekend before break and first after break
t_pre  = BREAK_START_DATE - 1
t_post = BREAK_END_DATE   + 1

In [12]:
# z[i,j] can be 1 only if both i at t_pre and j at t_post
z_cons1 = gp.Equation(m, domain=[i, j])
z_cons1[i, j] = z[i, j] <= y[i, t_pre]

z_cons2 = gp.Equation(m, domain=[i, j])
z_cons2[i, j] = z[i, j] <= y[j, t_post]

z_cons3 = gp.Equation(m, domain=[i, j])
z_cons3[i, j] = z[i, j] >= y[i, t_pre] + y[j, t_post] - 1

In [13]:
# Objective
total_emissions = (
    # 1. HQ → AUS
    gp.Sum(Team, hq_emissions[Team, FIRST])
    
    # 2. Sum of all consecutive race emissions
    + gp.Sum([i,j], r_emissions[i,j] * x[i,j])
    
    # 3. Last race → HQ
    + gp.Sum(Team, hq_emissions[Team, LAST])

    # ---- SUMMER BREAK ADJUSTMENTS ----

    # 4. REMOVE emission of arc i* → j*
    - gp.Sum([i, j], 
          r_emissions[i, j] 
          * z[i,j]
      )

    # 5. ADD i* → HQ
    + gp.Sum(Circuit, hq_emissions["ALL", Circuit] * y[Circuit, t_pre])

    # 6. ADD HQ → j*
    + gp.Sum(Circuit, hq_emissions["ALL", Circuit] * y[Circuit, t_post])

    # ---- LOOP CLOSURE ADJUSTMENT ----

    # 7. REMOVE ABD → AUS arc if present
    - r_emissions[LAST, FIRST] * x[LAST, FIRST]
)

model = gp.Model(
    m,
    name="f1_calendar",
    equations=m.getEquations(),
    sense=gp.Sense.MIN,
    problem=gp.Problem.MIP,
    objective=total_emissions
)

model.solve()

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,79822.181707,3358,2160,MIP,CPLEX,0.753


In [14]:
df = x.records                  # adjacency matrix in long form
df2 = u.records                 # each race with its order (u variable)
df1 = df[df['level'] == 1]      # edges actually used (x[i,j] = 1)


In [15]:
order_df = df2.sort_values("level").reset_index(drop=True)
edges = df1[['Circuit_0', 'Circuit_1']].reset_index(drop=True)

import plotly.graph_objects as go

# Step 1: Race order
order_df = df2.sort_values("level").reset_index(drop=True)

# Step 2: Edges (from x variable)
edges = df1[['Circuit_0', 'Circuit_1']].reset_index(drop=True)

# Step 3: Bring coordinates
coords = circuits_df[['circuit_id', 'latitude', 'longitude']]

edges = (
    edges.merge(coords, left_on='Circuit_0', right_on='circuit_id')
         .rename(columns={'latitude':'lat_from', 'longitude':'lon_from'})
         .drop(columns=['circuit_id'])
         .merge(coords, left_on='Circuit_1', right_on='circuit_id')
         .rename(columns={'latitude':'lat_to', 'longitude':'lon_to'})
         .drop(columns=['circuit_id'])
)

# Step 4: Create Plotly figure
fig = go.Figure()

# Draw each travel segment
for _, row in edges.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row['lon_from'], row['lon_to']],
        lat=[row['lat_from'], row['lat_to']],
        mode='lines',
        line=dict(width=2, color='red'),
        opacity=0.7,
        name=f"{row['Circuit_0']} → {row['Circuit_1']}"
    ))

# Draw race locations as points
fig.add_trace(go.Scattergeo(
    lon=coords['longitude'],
    lat=coords['latitude'],
    mode='markers',
    marker=dict(size=6, color="blue"),
    text=coords['circuit_id'],
    name="Circuits"
))

fig.update_layout(
    title="F1 Calendar Travel Route",
    geo=dict(
        projection_type="natural earth",
        showcountries=True,
        landcolor="rgb(240, 240, 240)",
    ),
    height=650
)

fig.show()


In [16]:
import plotly.graph_objects as go
import numpy as np

# Step 1: Race order
order_df = df2.sort_values("level").reset_index(drop=True)

# Step 2: Edges
edges = df1[['Circuit_0', 'Circuit_1']].reset_index(drop=True)

# Step 3: Coordinates
coords = circuits_df[['circuit_id', 'latitude', 'longitude']]

edges = (
    edges.merge(coords, left_on='Circuit_0', right_on='circuit_id')
         .rename(columns={'latitude':'lat_from', 'longitude':'lon_from'})
         .drop(columns=['circuit_id'])
         .merge(coords, left_on='Circuit_1', right_on='circuit_id')
         .rename(columns={'latitude':'lat_to', 'longitude':'lon_to'})
         .drop(columns=['circuit_id'])
)

fig = go.Figure()

# Function to create arrowhead
def arrowhead(lat_from, lon_from, lat_to, lon_to, scale=0.15):
    """Return a small arrowhead point slightly before the destination."""

    # Direction vector
    dlat = lat_to - lat_from
    dlon = lon_to - lon_from

    # Shorten vector for arrowhead base
    lat_arrow = lat_to - scale * dlat
    lon_arrow = lon_to - scale * dlon

    return lat_arrow, lon_arrow

# Draw route lines + arrowheads
for _, row in edges.iterrows():

    # Main route line
    fig.add_trace(go.Scattergeo(
        lon=[row['lon_from'], row['lon_to']],
        lat=[row['lat_from'], row['lat_to']],
        mode='lines',
        line=dict(width=2, color='red'),
        opacity=0.8,
        name=f"{row['Circuit_0']} → {row['Circuit_1']}"
    ))

    # Arrowhead segment
    lat_arrow, lon_arrow = arrowhead(row['lat_from'], row['lon_from'],
                                     row['lat_to'], row['lon_to'])

    fig.add_trace(go.Scattergeo(
        lon=[lon_arrow, row['lon_to']],
        lat=[lat_arrow, row['lat_to']],
        mode='lines',
        line=dict(width=4, color='red'),
        opacity=1.0,
        showlegend=False
    ))

# Circuit nodes
fig.add_trace(go.Scattergeo(
    lon=coords['longitude'],
    lat=coords['latitude'],
    mode='markers+text',
    marker=dict(size=8, color="blue"),
    text=coords['circuit_id'],
    textposition="top center",
    name="Circuits"
))

fig.update_layout(
    title="F1 Calendar Travel Route (with Direction Arrows)",
    geo=dict(
        projection_type="natural earth",
        showcountries=True,
        landcolor="rgb(240, 240, 240)",
    ),
    height=650
)

fig.show()


In [17]:
pd.merge(df1, df2, left_on = 'Circuit_0', right_on = 'Circuit').sort_values(by='level_y')

,Circuit_0,Circuit_1,level_x,marginal_x,lower_x,upper_x,scale_x,Circuit,level_y,marginal_y,lower_y,upper_y,scale_y
0,BAH,AZE,1.0,-0.000000,0.0,1.0,1.0,BAH,1.0,0.0,1.0,1.0,1.0
17,AZE,AUT,1.0,-0.000000,0.0,1.0,1.0,AZE,2.0,-0.0,2.0,24.0,1.0
10,AUT,MONZ,1.0,452.176942,0.0,1.0,1.0,AUT,3.0,0.0,2.0,24.0,1.0
15,MONZ,MEX,1.0,-0.000000,0.0,1.0,1.0,MONZ,4.0,0.0,2.0,24.0,1.0
20,MEX,USA_LVG,1.0,16426.372293,0.0,1.0,1.0,MEX,5.0,0.0,2.0,24.0,1.0
19,USA_LVG,USA_COT,1.0,-0.000000,0.0,1.0,1.0,USA_LVG,6.0,0.0,2.0,24.0,1.0
18,USA_COT,SIN,1.0,-0.000000,0.0,1.0,1.0,USA_COT,7.0,0.0,2.0,24.0,1.0
16,SIN,SAU,1.0,-0.000000,0.0,1.0,1.0,SIN,8.0,0.0,2.0,24.0,1.0
1,SAU,QAT,1.0,-0.000000,0.0,1.0,1.0,SAU,9.0,0.0,2.0,24.0,1.0
22,QAT,NED,1.0,-0.000000,0.0,1.0,1.0,QAT,10.0,0.0,2.0,24.0,1.0
